# Gold Price Prediction using Random Forest

In [1]:
import yfinance as yf
import pandas as pd
import numpy as np
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error
import matplotlib.pyplot as plt

# Download Data
DATA_FILE = 'gold_price_data.csv'
if not os.path.exists(DATA_FILE):
    print('Downloading historical gold prices...')
    gold_data = yf.download('GC=F', start='2015-01-01', end='2023-12-31')
    gold_data.to_csv(DATA_FILE)
else:
    print(f'{DATA_FILE} already exists. Skipping download.')
    gold_data = pd.read_csv(DATA_FILE, index_col='Date', parse_dates=True)

gold_data.head()

gold_price_data.csv already exists. Skipping download.


ValueError: 'Date' is not in list

In [ ]:
# Preprocess and Train
df = gold_data.copy().dropna()
df['Prev_Close'] = df['Close'].shift(1)
df['MA_5'] = df['Close'].rolling(window=5).mean().shift(1)
df['MA_10'] = df['Close'].rolling(window=10).mean().shift(1)
df = df.dropna()

features = ['Open', 'High', 'Low', 'Volume', 'Prev_Close', 'MA_5', 'MA_10']
X = df[features]
y = df['Close']

split_index = int(len(X) * 0.8)
X_train, X_test = X.iloc[:split_index], X.iloc[split_index:]
y_train, y_test = y.iloc[:split_index], y.iloc[split_index:]

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

predictions = model.predict(X_test)
rmse = np.sqrt(mean_squared_error(y_test, predictions))
mae = mean_absolute_error(y_test, predictions)
print(f'RMSE: {rmse:.2f}')
print(f'MAE: {mae:.2f}')

importances = model.feature_importances_
indices = np.argsort(importances)
plt.figure(figsize=(10, 6))
plt.title('Feature Importances')
plt.barh(range(len(indices)), importances[indices], color='b', align='center')
plt.yticks(range(len(indices)), [X.columns[i] for i in indices])
plt.xlabel('Relative Importance')
plt.show()